In [1]:
import math
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

In [2]:
# 1. SPATIAL FEATURE EXTRACTOR

class MobileNetFeatureExtractor(nn.Module):
    def __init__(self, freeze_backbone=True):
        super(MobileNetFeatureExtractor, self).__init__()
        # 1. Loading pretrained backbone weights
        self.backbone = models.mobilenet_v2(weights='DEFAULT')
        # 2. Dropping the standard 1000-class head
        self.backbone.classifier = nn.Identity() 
        
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
                
    def forward(self, x):
        # Input Check Contract: x must be (8, 32, 3, 224, 224)
        batch_size, timesteps, C, H, W = x.shape
        
        # Flatten Time: Collapse Batch + Timesteps to process 256 images in parallel
        flat_images = x.view(batch_size * timesteps, C, H, W) 
        
        # CNN Forward Pass -> Output shape: (256, 1280)
        flat_features = self.backbone(flat_images) 
        
        # Unflatten Time: Reconstruct sequential format for Transformer -> (8, 32, 1280)
        # @Member_4: This is the exact tensor shape your Transformer Encoder expects!
        sequence_features = flat_features.view(batch_size, timesteps, 1280) 
        return sequence_features

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x

class SignSpeechTransformer(nn.Module):
    def __init__(self, num_classes=100, d_model=1280, nhead=8, num_layers=2, dim_feedforward=512, dropout=0.1):
        super(SignSpeechTransformer, self).__init__()
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        # Updated to 100 classes
        self.classifier = nn.Linear(d_model, num_classes)
        
    def forward(self, x):
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)
        logits = self.classifier(x)
        return logits

class SignSpeechBiLSTM(nn.Module):
    def __init__(self, num_classes=100, input_dim=1280, hidden_dim=256, num_layers=2, dropout=0.2):
        super(SignSpeechBiLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True, dropout=dropout if num_layers > 1 else 0
        )
        # Updated to 100 classes (hidden_dim * 2 due to bidirectional)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)
        
    def forward(self, x):
        lstm_out, (h_n, c_n) = self.lstm(x)
        out = lstm_out[:, -1, :]
        logits = self.classifier(out)
        return logits

In [4]:
class SignSpeechCompletePipeline(nn.Module):
    def __init__(self, temporal_type='transformer', num_classes=100):
        super(SignSpeechCompletePipeline, self).__init__()
        self.spatial_extractor = MobileNetFeatureExtractor(freeze_backbone=True)
        
        self.temporal_type = temporal_type.lower()
        if self.temporal_type == 'transformer':
            self.temporal_model = SignSpeechTransformer(num_classes=num_classes)
        elif self.temporal_type == 'bilstm':
            self.temporal_model = SignSpeechBiLSTM(num_classes=num_classes)
        else:
            raise ValueError("Choose either 'transformer' or 'bilstm'")
            
    def forward(self, x):
        spatial_features = self.spatial_extractor(x)
        logits = self.temporal_model(spatial_features)
        return logits

In [5]:
def train_and_evaluate_model(model, train_loader, val_loader, epochs=10, lr=0.001, device='cuda'):
    """
    Comprehensive Training Loop for SignSpeech Framework (M4 Task)
    Trains the complete pipeline and evaluates on validation data.
    """
    model = model.to(device)
    
    # 1. Define Loss Function and Optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    print(f"Starting Training on Device: {device}...")
    
    for epoch in range(epochs):
        # ── TRAINING PHASE ──
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for batch_videos, batch_labels in train_loader:
            # Move data to target hardware (GPU/CPU)
            batch_videos = batch_videos.to(device) # Shape: [B, T, C, H, W]
            batch_labels = batch_labels.to(device) # Shape: [B]
            
            # Forward pass
            optimizer.zero_grad()
            logits = model(batch_videos) # Shape: [B, 29]
            loss = criterion(logits, batch_labels)
            
            # Backward pass and optimization
            loss.backward()
            optimizer.step()
            
            # Track metrics
            running_loss += loss.item() * batch_videos.size(0)
            _, predicted = torch.max(logits, dim=1)
            total_train += batch_labels.size(0)
            correct_train += (predicted == batch_labels).sum().item()
            
        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = (correct_train / total_train) * 100
        
        # ── VALIDATION PHASE ──
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0
        
        with torch.no_grad():
            for batch_videos, batch_labels in val_loader:
                batch_videos = batch_videos.to(device)
                batch_labels = batch_labels.to(device)
                
                logits = model(batch_videos)
                loss = criterion(logits, batch_labels)
                
                val_loss += loss.item() * batch_videos.size(0)
                _, predicted = torch.max(logits, dim=1)
                total_val += batch_labels.size(0)
                correct_val += (predicted == batch_labels).sum().item()
                
        epoch_val_loss = val_loss / len(val_loader.dataset)
        epoch_val_acc = (correct_val / total_val) * 100
        
        # Print metrics for the current epoch
        print(f"Epoch [{epoch+1}/{epochs}] -> "
              f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}% || "
              f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")
        
    print("Training process finished completely.")
    return model

In [6]:
# Select Hardware Accelerator
device_target = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. Train Proposed Transformer Pipeline (Ensure train_loader & val_loader are provided by M1)
print("--- Training Proposed Transformer Pipeline ---")
transformer_pipeline = SignSpeechCompletePipeline(temporal_type='transformer', num_classes=100)
# To execute training, uncomment the line below once data loaders are ready:
# trained_transformer = train_and_evaluate_model(transformer_pipeline, train_loader, val_loader, device=device_target)

# Save best model checkpoint weights for Member 5
torch.save(transformer_pipeline.state_dict(), 'signspeech_transformer_best.pth')
print("Transformer weights saved successfully.")

--- Training Proposed Transformer Pipeline ---
Transformer weights saved successfully.


In [7]:
if __name__ == "__main__":
    mock_video_batch = torch.randn(8, 32, 3, 224, 224)
    print(f"Target Input Tensor Shape: {mock_video_batch.shape}\n")
    
    # Test Transformer Pipeline
    transformer_pipeline = SignSpeechCompletePipeline(temporal_type='transformer', num_classes=100)
    output_transformer = transformer_pipeline(mock_video_batch)
    print(f"Proposed Transformer Pipeline Output Shape: {output_transformer.shape}") # Expected: [8, 100]
    
    # Test BiLSTM Pipeline
    bilstm_pipeline = SignSpeechCompletePipeline(temporal_type='bilstm', num_classes=100)
    output_bilstm = bilstm_pipeline(mock_video_batch)
    print(f"Baseline BiLSTM Pipeline Output Shape: {output_bilstm.shape}") # Expected: [8, 100]

Target Input Tensor Shape: torch.Size([8, 32, 3, 224, 224])

Proposed Transformer Pipeline Output Shape: torch.Size([8, 100])
Baseline BiLSTM Pipeline Output Shape: torch.Size([8, 100])


In [8]:
# Model Parameter Profiling Helper for Ablation Report
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
        
print("--- Trainable Parameters Count (Ablation Metrics) ---")
print(f"Transformer Encoder Layer: {count_parameters(transformer_pipeline.temporal_model)} parameters")
print(f"BiLSTM Baseline Layer:     {count_parameters(bilstm_pipeline.temporal_model)} parameters")

--- Trainable Parameters Count (Ablation Metrics) ---
Transformer Encoder Layer: 15880804 parameters
BiLSTM Baseline Layer:     4778084 parameters
